### Dummy Data Creation (Temporary)

This cell creates a placeholder `content_refresh_anonymized.csv` file to prevent `FileNotFoundError`. Please replace this dummy file with your actual dataset in `data/raw/` once available.

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayak-D/FlyRank---Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I rank pages for refresh review using signals that a page is still visible, old enough to need a new update, and underperforming on clicks for its search position. The rule is designed to surface actionable review candidates without using future or product-derived flags.

Reason codes:
- `stale_visible_page` — high impressions and long time since last update
- `low_ctr_visible_page` — strong visibility but CTR below the position-specific expectation
- `refresh_review` — general review candidate when the page is still worth checking

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

candidate_paths = [
    Path.cwd() / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
]

path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError(
        'Could not find data/raw/content_refresh_anonymized.csv.\n'
        'Expected it under the repo root or one level above the notebook folder.'
    )

print('Loading dataset from:', path)
df = pd.read_csv(path)

print('\nDataset rows:', len(df))
print('Columns available:', sorted([c for c in df.columns if c in ['content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'trend_direction']]))

# Signal 1: staleness behind the refresh-style rule.
staleness_bins = [0, 30, 90, 180, 365, 10_000]
staleness_labels = ['0-29d', '30-89d', '90-179d', '180-364d', '365+d']
df['staleness_bucket'] = pd.cut(df['days_since_last_update'].fillna(0), bins=staleness_bins, labels=staleness_labels, right=False)

staleness_summary = (
    df.groupby('staleness_bucket', observed=True)
    .agg(
        n=('content_id', 'size'),
        median_impressions=('impressions_90d', 'median'),
        declining_rate=('trend_direction', lambda x: (x.str.lower() == 'down').mean()),
    )
    .reset_index()
)
print('\nStaleness buckets')
print(staleness_summary.to_string(index=False))

# Signal 2: CTR versus position behind the CTR-fix logic.
position_bins = [0, 3, 6, 11, 21, 51, 10_000]
position_labels = ['1-2', '3-5', '6-10', '11-20', '21-50', '50+']
df['position_bucket'] = pd.cut(df['avg_position'].fillna(999), bins=position_bins, labels=position_labels, right=False)

ctr_expectation = (
    df.groupby('position_bucket', observed=True)['ctr']
    .median()
    .rename('expected_ctr')
)

ctr_summary = (
    df.join(ctr_expectation, on='position_bucket')
    .assign(ctr_gap=lambda x: x['expected_ctr'] - x['ctr'])
    .groupby('position_bucket', observed=True)
    .agg(
        n=('content_id', 'size'),
        median_ctr=('ctr', 'median'),
        mean_ctr=('ctr', 'mean'),
        below_expectation=('ctr_gap', lambda x: (x > 0.01).mean()),
        declining_rate=('trend_direction', lambda x: (x.str.lower() == 'down').mean()),
    )
    .reset_index()
)
print('\nCTR-by-position buckets')
print(ctr_summary.to_string(index=False))

visible_top = df.loc[df['avg_position'] <= 20, :].copy()
visible_top = visible_top.join(ctr_expectation, on='position_bucket')
low_ctr_visible = visible_top.loc[visible_top['ctr'] < visible_top['expected_ctr'] - 0.01]
print(f"\nVisible pages (pos <= 20): {len(visible_top)}")
print(f"Low-CTR visible pages: {len(low_ctr_visible)} ({len(low_ctr_visible) / len(visible_top):.1%} of visible pages)")

# One-word verdicts for the two signals.
baseline_decline_rate = (df['trend_direction'].str.lower() == 'down').mean()
staleness_verdict = 'CONFIRMED' if staleness_summary.loc[staleness_summary['staleness_bucket'] == '180-364d', 'declining_rate'].iloc[0] >= baseline_decline_rate else 'MIXED'
ctr_verdict = 'CONFIRMED' if len(low_ctr_visible) >= 0.10 * len(visible_top) else 'MIXED'

print('\nSignal verdicts:')
print('staleness:', staleness_verdict)
print('ctr_vs_position:', ctr_verdict)

Loading dataset from: /content/data/raw/content_refresh_anonymized.csv

Dataset rows: 100
Columns available: ['avg_position', 'client_id', 'content_id', 'ctr', 'days_since_last_update', 'impressions_90d', 'trend_direction']

Staleness buckets
staleness_bucket  n  median_impressions  declining_rate
           0-29d  4              5116.0        0.500000
          30-89d  9              5250.0        0.666667
         90-179d 19              4994.0        0.578947
        180-364d 41              5282.0        0.390244
           365+d 27              7005.0        0.407407

CTR-by-position buckets
position_bucket  n  median_ctr  mean_ctr  below_expectation  declining_rate
            1-2  4    0.022836  0.023558           0.250000        0.250000
            3-5  7    0.026036  0.026993           0.142857        0.428571
           6-10  7    0.020298  0.018204           0.142857        0.428571
          11-20 23    0.025554  0.024938           0.260870        0.608696
          21-50 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
output_path = Path.cwd().parent / 'outputs' / 'baseline_action_score.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)

# Use only observable signals in the score.
queue = df.copy()
queue['expected_ctr'] = queue['position_bucket'].map(ctr_expectation)
queue['visibility_score'] = np.log1p(queue['impressions_90d']) / np.log1p(queue['impressions_90d']).max()
queue['staleness_score'] = queue['days_since_last_update'].clip(lower=0, upper=365) / 365
queue['position_score'] = np.where(queue['avg_position'] > 0, ((50 - queue['avg_position'].clip(upper=50)) / 50), 0.0)
queue['ctr_opportunity_score'] = np.where(
    (queue['avg_position'] <= 20)
    & (queue['ctr'] >= 0)
    & (queue['ctr'] < queue['expected_ctr'] - 0.01),
    ((queue['expected_ctr'] - queue['ctr']) / queue['expected_ctr']).clip(0, 1),
    0.0,
)

queue['baseline_action_score'] = (
    0.35 * queue['visibility_score']
    + 0.30 * queue['staleness_score']
    + 0.25 * queue['position_score']
    + 0.10 * queue['ctr_opportunity_score']
).clip(0, 1)

queue['reason_code'] = np.select(
    [
        (queue['days_since_last_update'] >= 180) & (queue['impressions_90d'] >= 500),
        (queue['avg_position'] <= 20) & (queue['ctr'] < queue['expected_ctr'] - 0.01) & (queue['impressions_90d'] >= 250),
    ],
    ['stale_visible_page', 'low_ctr_visible_page'],
    default='refresh_review',
)

queue['action_label'] = np.select(
    [
        queue['reason_code'] == 'low_ctr_visible_page',
        queue['reason_code'] == 'stale_visible_page',
    ],
    ['refresh_and_review_ctr', 'refresh'],
    default='monitor',
)

queue['baseline_rank'] = queue['baseline_action_score'].rank(method='first', ascending=False).astype(int)
queue = queue.sort_values(['baseline_action_score', 'visibility_score', 'staleness_score'], ascending=[False, False, False])
queue['baseline_rank'] = range(1, len(queue) + 1)

out_columns = [
    'content_id',
    'client_id',
    'baseline_rank',
    'baseline_action_score',
    'reason_code',
    'action_label',
    'impressions_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'position_bucket',
    'visibility_score',
    'staleness_score',
    'position_score',
    'ctr_opportunity_score',
]
out = queue[out_columns]
out.to_csv(output_path, index=False)

print(f'Wrote baseline queue: {output_path}')
print('Top-5 rows:')
print(out.head(5).to_string(index=False))

Wrote baseline queue: /outputs/baseline_action_score.csv
Top-5 rows:
content_id client_id  baseline_rank  baseline_action_score        reason_code action_label  impressions_90d  avg_position      ctr  days_since_last_update position_bucket  visibility_score  staleness_score  position_score  ctr_opportunity_score
     id_32  client_2              1               0.920059 stale_visible_page      refresh             9015      6.848105 0.008509                     479            6-10          0.989195         1.000000        0.863038               0.580817
     id_61  client_1              2               0.888961 stale_visible_page      refresh             3955      2.573072 0.006145                     321             1-2          0.899715         0.879452        0.948539               0.730899
     id_51  client_1              3               0.872005 stale_visible_page      refresh             8478      4.375737 0.024819                     419             3-5          0.982524        

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = out.head(20).copy()
review_lines = []
for _, row in top20.iterrows():
    note = f"Rank {row['baseline_rank']}: {row['action_label']} ({row['reason_code']})"
    note += f" — {row['impressions_90d']} impressions, position {row['avg_position']:.1f}, ctr {row['ctr']:.3f}."
    if row['reason_code'] == 'stale_visible_page':
        wrong = 'Wrong if the page was actually refreshed recently or if impressions are noisy.'
    elif row['reason_code'] == 'low_ctr_visible_page':
        wrong = 'Wrong if the CTR is normal for this position or if the position expectation is misleading.'
    else:
        wrong = 'Wrong if the page has no real demand or if a refresh would not change its search visibility.'
    review_lines.append(f"{note} {wrong}")

print('\nTop 20 review lines:')
print('\n'.join(review_lines))


Top 20 review lines:
Rank 1: refresh (stale_visible_page) — 9015 impressions, position 6.8, ctr 0.009. Wrong if the page was actually refreshed recently or if impressions are noisy.
Rank 2: refresh (stale_visible_page) — 3955 impressions, position 2.6, ctr 0.006. Wrong if the page was actually refreshed recently or if impressions are noisy.
Rank 3: refresh (stale_visible_page) — 8478 impressions, position 4.4, ctr 0.025. Wrong if the page was actually refreshed recently or if impressions are noisy.
Rank 4: refresh (stale_visible_page) — 9371 impressions, position 11.7, ctr 0.012. Wrong if the page was actually refreshed recently or if impressions are noisy.
Rank 5: refresh (stale_visible_page) — 5926 impressions, position 3.6, ctr 0.048. Wrong if the page was actually refreshed recently or if impressions are noisy.
Rank 6: refresh (stale_visible_page) — 7424 impressions, position 4.1, ctr 0.014. Wrong if the page was actually refreshed recently or if impressions are noisy.
Rank 7: ref

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
weak = out.loc[(out['baseline_rank'] <= 20) & (out['impressions_90d'] < 250)]
print('Weak top-20 picks by low volume:')
print(weak[['baseline_rank', 'action_label', 'reason_code', 'impressions_90d', 'avg_position', 'ctr']].to_string(index=False))

leakage_columns = [
    c for c in out.columns
    if c in ['health_score', 'priority_score', 'action_type', 'refresh_flag', 'refresh_tier', 'trend_direction', 'is_declining_label']
]
print('\nLeakage columns found:', leakage_columns)
print('No product flags or future-window fields are used in the score or reason code.')

Weak top-20 picks by low volume:
Empty DataFrame
Columns: [baseline_rank, action_label, reason_code, impressions_90d, avg_position, ctr]
Index: []

Leakage columns found: []
No product flags or future-window fields are used in the score or reason code.
